# Análise quantitativa de casos ruins

## Objetivo

Analisa quantitativamente os casos ruins e compara seu comportamento entre mid e high resolution.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Any, Optional
from utils.project.results.schema import normalize_result_status, result_status_label_pt


def compare_shared_bad_cases(
    df_mid: Optional[pd.DataFrame],
    df_high: Optional[pd.DataFrame],
    subset_label: str,
    dice_threshold: float = 0.30,
) -> Optional[dict]:
    """Compara casos ruins compartilhados entre resultados mid e high."""
    if df_mid is None or df_high is None or df_mid.empty or df_high.empty:
        return None

    id_col_mid = "IMG_ID" if "IMG_ID" in df_mid.columns else "img_id"
    id_col_high = "IMG_ID" if "IMG_ID" in df_high.columns else "img_id"

    def _status_ids(df, status_label, id_col):
        if "status" not in df.columns:
            return set()
        return set(
            df.loc[df["status"].eq(status_label).fillna(False), id_col]
            .dropna()
            .astype(int)
        )

    def _low_dice_mask(df):
        return (
            pd.to_numeric(df.get("dice_artery"), errors="coerce") < dice_threshold
        ).fillna(False)

    ids_low_mid = set(
        df_mid.loc[_low_dice_mask(df_mid), id_col_mid].dropna().astype(int)
    )
    ids_low_high = set(
        df_high.loc[_low_dice_mask(df_high), id_col_high].dropna().astype(int)
    )
    ids_low_both = ids_low_mid & ids_low_high

    status_groups = [
        ("Óstios não encontrados em ambos", "not_found"),
        ("Um correto em ambos", "one_correct"),
        ("Nenhum correto em ambos", "none_correct"),
        ("Erro em ambos", "error"),
    ]

    status_intersections = {}
    for display_name, status_label in status_groups:
        ids_mid = _status_ids(df_mid, status_label, id_col_mid)
        ids_high = _status_ids(df_high, status_label, id_col_high)
        status_intersections[display_name] = ids_mid & ids_high

    compare_rows = [
        {"tipo": f"Dice < {dice_threshold} em ambos", "quantidade": len(ids_low_both)}
    ]
    compare_rows.extend(
        {"tipo": display_name, "quantidade": len(ids)}
        for display_name, ids in status_intersections.items()
    )
    compare_df = pd.DataFrame(compare_rows)

    plt.figure(figsize=(9.5, 4.8))
    ax_cmp = sns.barplot(data=compare_df, x="tipo", y="quantidade", color="#4C78A8")
    ax_cmp.set_title(f"Intersecao Mid vs High por IMG_ID - {subset_label}")
    ax_cmp.set_xlabel("Tipo de intersecao")
    ax_cmp.set_ylabel("Quantidade de imagens")
    ax_cmp.tick_params(axis="x", rotation=15)
    ax_cmp.grid(axis="y", alpha=0.3)
    for container in ax_cmp.containers:
        ax_cmp.bar_label(container, fmt="%d", padding=3)
    plt.tight_layout()
    plt.show()

    return {"ids_low_both": ids_low_both, "status_intersections": status_intersections}


def plot_bad_dice_indicator(
    df_mid_bad: Optional[pd.DataFrame],
    df_high_bad: Optional[pd.DataFrame],
    subset_label: str,
    summarize_bad_dice_fn: Any,
    dice_threshold: float = 0.3,
) -> pd.DataFrame:
    """Plota o Dice dos casos ruins incluindo ou não sucessos com Dice baixo."""
    mid_stats = summarize_bad_dice_fn(df_mid_bad, dice_threshold=dice_threshold)
    high_stats = summarize_bad_dice_fn(df_high_bad, dice_threshold=dice_threshold)

    rows = []
    for resolution_label, stats in [
        ("Mid Res", mid_stats),
        ("High Res", high_stats),
    ]:
        if stats["n_with_low_dice"] == 0:
            continue
        rows.append(
            {
                "resolution": resolution_label,
                "com_dice_baixo_ok": stats["mean_with_low_dice"],
                "sem_dice_baixo_ok": stats["mean_without_low_dice"],
                "n_total_dice": stats["n_with_low_dice"],
                "n_removidos_dice_baixo_ok": stats["n_low_dice_correct"],
            }
        )

    indicator_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(10, 4.8))
    if indicator_df.empty:
        ax.text(0.5, 0.5, "Sem dados de Dice para plotar", ha="center", va="center")
        ax.axis("off")
        plt.tight_layout()
        return indicator_df

    x = np.arange(len(indicator_df))
    width = 0.35

    bars_with = ax.bar(
        x - width / 2,
        indicator_df["com_dice_baixo_ok"],
        width,
        label=f"Com Dice < {int(dice_threshold * 100)}% (óstio correto)",
        color="#4C78A8",
    )
    bars_without = ax.bar(
        x + width / 2,
        indicator_df["sem_dice_baixo_ok"],
        width,
        label=f"Sem Dice < {int(dice_threshold * 100)}% (óstio correto)",
        color="#F58518",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(indicator_df["resolution"].tolist())
    ax.set_ylim(0, 1)
    ax.set_ylabel("Dice Score médio")
    ax.set_title(
        f"Indicador de Dice dos casos ruins - {subset_label}", fontweight="bold"
    )
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    for bars in (bars_with, bars_without):
        for bar in bars:
            height = bar.get_height()
            if pd.notna(height):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height + 0.015,
                    f"{height:.4f}",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                )

    plt.tight_layout()
    return indicator_df


def change_status_label_for_plot(status: Any) -> str:
    """Normaliza rótulos de status para visualização em gráfico."""
    normalized = normalize_result_status(status)
    if normalized in {"error", "not_found"}:
        return "óstios não encontrados"
    if normalized in {"both_tolerable", "both_correct"}:
        return "baixo Dice score"
    return result_status_label_pt(normalized)


def plot_bad_cases_by_subset(
    df_mid: Optional[pd.DataFrame],
    df_high: Optional[pd.DataFrame],
    df_mid_bad: Optional[pd.DataFrame],
    df_high_bad: Optional[pd.DataFrame],
    subset_label: str,
) -> None:
    """Plota distribuição de status dos casos ruins por subconjunto."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if df_mid is not None and not df_mid.empty:
        mid_status = df_mid_bad["status"].fillna("sem status").value_counts()
        mid_status.index = mid_status.index.map(change_status_label_for_plot)
        mid_status = mid_status.groupby(level=0).sum()
        sns.barplot(
            x=mid_status.index, y=mid_status.values, color="#4C78A8", ax=axes[0]
        )
        axes[0].set_xlabel("Status")
        axes[0].set_ylabel("Quantidade de casos")
        axes[0].set_title(
            f"Distribuicao de status dos casos ruins - Mid Res ({subset_label})"
        )
        axes[0].tick_params(axis="x", rotation=45)
        for patch in axes[0].patches:
            height = patch.get_height()
            axes[0].text(
                patch.get_x() + patch.get_width() / 2.0,
                height,
                f"{int(height)}",
                ha="center",
                va="bottom",
            )
    else:
        axes[0].text(
            0.5,
            0.5,
            f"Sem dados de {subset_label.lower()} para Mid Res",
            ha="center",
            va="center",
            transform=axes[0].transAxes,
        )
        axes[0].set_title(
            f"Distribuicao de status dos casos ruins - Mid Res ({subset_label})"
        )

    if df_high is not None and not df_high.empty:
        high_status = df_high_bad["status"].fillna("sem status").value_counts()
        high_status.index = high_status.index.map(change_status_label_for_plot)
        high_status = high_status.groupby(level=0).sum()
        sns.barplot(
            x=high_status.index, y=high_status.values, color="#F58518", ax=axes[1]
        )
        axes[1].set_xlabel("Status")
        axes[1].set_ylabel("Quantidade de casos")
        axes[1].set_title(
            f"Distribuicao de status dos casos ruins - High Res ({subset_label})"
        )
        axes[1].tick_params(axis="x", rotation=45)
        for patch in axes[1].patches:
            height = patch.get_height()
            axes[1].text(
                patch.get_x() + patch.get_width() / 2.0,
                height,
                f"{int(height)}",
                ha="center",
                va="bottom",
            )
    else:
        axes[1].text(
            0.5,
            0.5,
            f"Sem dados de {subset_label.lower()} para High Res\n(Em preparacao)",
            ha="center",
            va="center",
            transform=axes[1].transAxes,
            fontsize=11,
            color="gray",
        )
        axes[1].set_title(
            f"Distribuicao de status dos casos ruins - High Res ({subset_label})"
        )
        axes[1].set_xticks([])
        axes[1].set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
from utils.project.runtime.notebook_env import get_default_split_paths
import pandas as pd

import utils.comparison_utils.bad_cases as cmp_bad_cases
import utils.visualization as viz

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
        "savefig.dpi": 300,
    },
)
plt.close("all")

SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)

VALID_SPLITS = ("train", "val", "test")

## Carregamento de dados

### Dados dos casos ruins (bad_tests)

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

In [ ]:
bad_cases_split = "test"  # options: "train", "val" or "test"
BAD_CASES_EXPORT_DIR = REPO_ROOT / "output/segmentation/analysis/bad_cases"

if bad_cases_split not in VALID_SPLITS:
    raise ValueError("bad_cases_split must be 'train', 'val' or 'test'.")
if bad_cases_split not in SPLIT_PATHS_BY_RESOLUTION["mid_res"]:
    raise ValueError(f"Split '{bad_cases_split}' is not available for mid_res.")

# Cada split é carregado, filtrado e exportado uma única vez.
bad_cases_by_split = {
    split: cmp_bad_cases.prepare_bad_cases_for_subset(
        split_paths_by_resolution=SPLIT_PATHS_BY_RESOLUTION,
        split_name=split,
        output_dir=BAD_CASES_EXPORT_DIR,
        valid_splits=VALID_SPLITS,
    )
    for split in VALID_SPLITS
    if split in SPLIT_PATHS_BY_RESOLUTION["mid_res"]
}

selected_split_data = bad_cases_by_split[bad_cases_split]
df_mid_res = selected_split_data["df_mid"]
df_high_res = selected_split_data["df_high"]
df_mid_bad_res = selected_split_data["df_mid_bad"]
df_high_bad_res = selected_split_data["df_high_bad"]

bad_summary_rows = []
for resolution, summary_df, bad_df, export_info in (
    ("mid_res", df_mid_res, df_mid_bad_res, selected_split_data["mid_export"]),
    ("high_res", df_high_res, df_high_bad_res, selected_split_data["high_export"]),
):
    if summary_df is None:
        continue
    total_cases = int(summary_df.shape[0])
    bad_cases = int(bad_df.shape[0])
    bad_summary_rows.append(
        {
            "resolution": resolution,
            "subset": bad_cases_split,
            "total_cases": total_cases,
            "bad_cases": bad_cases,
            "bad_cases_percent": (
                100 * bad_cases / total_cases if total_cases > 0 else 0.0
            ),
            "csv_path": export_info["csv_path"],
            "json_path": export_info["json_path"],
        }
    )

if df_high_res is None:
    df_high_res = pd.DataFrame(columns=df_mid_res.columns)
    df_high_bad_res = pd.DataFrame(columns=df_mid_bad_res.columns)

bad_summary_df = pd.DataFrame(bad_summary_rows)

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Casos Ruins por Subset

### Validação (Val)

#### Carregamento

In [ ]:
val_data = bad_cases_by_split["val"]

df_mid_val = val_data["df_mid"]
df_high_val = val_data["df_high"]
df_mid_val_bad = val_data["df_mid_bad"]
df_high_val_bad = val_data["df_high_bad"]
val_mid_export = val_data["mid_export"]
val_high_export = val_data["high_export"]

#### Dice Score dos casos ruins (barra)

In [ ]:
val_bad_dice_indicator_df = plot_bad_dice_indicator(
    df_mid_val_bad,
    df_high_val_bad,
    "Validação",
    summarize_bad_dice_fn=cmp_bad_cases.summarize_bad_dice_with_threshold,
    dice_threshold=0.3,
)

#### Distribuição de casos ruins

In [ ]:
viz.bad_cases.plot_bad_cases_by_subset(
    df_mid_val,
    df_high_val,
    df_mid_val_bad,
    df_high_val_bad,
    "Validação",
)

#### Comparativo entre resoluções (Mid vs High)

In [ ]:
val_shared_bad = viz.bad_cases.compare_shared_bad_cases(
    df_mid_val, df_high_val, "Validação"
)

### Treino (Train)

#### Carregamento

In [ ]:
train_data = bad_cases_by_split["train"]

df_mid_train = train_data["df_mid"]
df_high_train = train_data["df_high"]
df_mid_train_bad = train_data["df_mid_bad"]
df_high_train_bad = train_data["df_high_bad"]
train_mid_export = train_data["mid_export"]
train_high_export = train_data["high_export"]

#### Dice Score dos casos ruins (barra)

In [ ]:
train_bad_dice_indicator_df = plot_bad_dice_indicator(
    df_mid_train_bad,
    df_high_train_bad,
    "Treino",
    summarize_bad_dice_fn=cmp_bad_cases.summarize_bad_dice_with_threshold,
    dice_threshold=0.3,
)
display(train_bad_dice_indicator_df)

#### Distribuição de casos ruins

In [ ]:
viz.bad_cases.plot_bad_cases_by_subset(
    df_mid_train, df_high_train, df_mid_train_bad, df_high_train_bad, "Treino"
)

#### Comparativo entre resoluções (Mid vs High)

In [ ]:
train_shared_bad = viz.bad_cases.compare_shared_bad_cases(
    df_mid_train, df_high_train, "Treino"
)

### Teste (Test)

#### Carregamento

In [ ]:
test_data = bad_cases_by_split["test"]

df_mid_test = test_data["df_mid"]
df_high_test = test_data["df_high"]
df_mid_test_bad = test_data["df_mid_bad"]
df_high_test_bad = test_data["df_high_bad"]
test_mid_export = test_data["mid_export"]
test_high_export = test_data["high_export"]

#### Dice Score dos casos ruins (barra)

In [ ]:
test_bad_dice_indicator_df = plot_bad_dice_indicator(
    df_mid_test_bad,
    df_high_test_bad,
    "Teste",
    summarize_bad_dice_fn=cmp_bad_cases.summarize_bad_dice_with_threshold,
    dice_threshold=0.3,
)
display(test_bad_dice_indicator_df)

#### Distribuição de casos ruins

In [ ]:
viz.bad_cases.plot_bad_cases_by_subset(
    df_mid_test, df_high_test, df_mid_test_bad, df_high_test_bad, "Teste"
)

#### Comparativo entre resoluções (Mid vs High)

In [ ]:
test_shared_bad = viz.bad_cases.compare_shared_bad_cases(
    df_mid_test, df_high_test, "Teste"
)

## Conclusão

As tabelas identificam os tipos de erro mais frequentes e os exames compartilhados entre resoluções, orientando a seleção da análise qualitativa.